In [1]:
# Install required packages
!pip install datasets gdown -q

In [2]:
import os
import json
from datasets import load_dataset

In [ ]:
# Step 1: Download dataset from Google Drive to Colab disk
dataset_path = "datasets/mind2web"

# Always show current state
print("=== Current working directory ===")
!pwd

print("\n=== Downloading from Google Drive ===")
os.makedirs("datasets", exist_ok=True)
!gdown --folder "1y4UAxRujRXVerPh0jkvP_RClrIRMxT3L" -O datasets --remaining-ok

print("\n=== What was downloaded? ===")
!ls -la datasets/

print("\n=== Contents of mind2web folder ===")
!ls -la datasets/mind2web/ 2>/dev/null || echo "Folder datasets/mind2web not found!"

In [ ]:
# Step 2: Find and load the dataset
print("=== Checking folder structure ===")

# Check both possible paths
paths_to_check = [
    "datasets/mind2web",
    "datasets/mind2web/mind2web"
]

dataset_path = None
for path in paths_to_check:
    state_file = os.path.join(path, "state.json")
    if os.path.exists(state_file):
        dataset_path = path
        print(f"✓ Found dataset at: {path}")
        break
    else:
        print(f"✗ No state.json at: {path}")

if dataset_path is None:
    print("\n⚠️ Could not find state.json! Searching...")
    !find datasets -name "state.json" 2>/dev/null
    raise FileNotFoundError("state.json not found in any expected location")

# Show contents
print(f"\n=== Files in {dataset_path} ===")
!ls {dataset_path}

# Count arrow files
arrow_count = len([f for f in os.listdir(dataset_path) if f.endswith('.arrow')])
print(f"\nArrow files: {arrow_count}/12")

if arrow_count < 12:
    print("⚠️ Missing arrow files! Download may have failed.")
    print("Try re-running with: !gdown --folder '1y4UAxRujRXVerPh0jkvP_RClrIRMxT3L' -O datasets --remaining-ok")
else:
    # Load dataset
    ds = load_from_disk(dataset_path)
    print(f"\n✓ Loaded {len(ds)} samples")
    
    # Show sample ground truth
    for i in range(3):
        gt = json.loads(ds[i]['conversations'][1]['value'])
        print(f"Sample {i}: {gt}")

In [ ]:
# Dataset statistics - count action types
from collections import Counter

action_counts = Counter()
for sample in ds:
    gt = json.loads(sample['conversations'][1]['value'])
    action_counts[gt['ACTION']] += 1

print("Action type distribution:")
for action, count in action_counts.most_common():
    print(f"  {action}: {count} ({count/len(ds)*100:.1f}%)")